# RetinaNet model for snowpole detection with LiDar and RGB datasets

### Import Packages

In [16]:
import torchvision
import torch
import torch.optim as optim
from torchvision.models.detection import RetinaNet_ResNet50_FPN_V2_Weights
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image # Or import cv2 if using OpenCV
import os
from transform import get_transforms, collate_fn
from dataloader import LidarDataset
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import time
import torchmetrics
from torchmetrics.detection import MeanAveragePrecision

### 1. Load the pre-trained model (using V2 weights, which are often better)

In [2]:

weights = RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT # Loads COCO pre-trained weights
model = torchvision.models.detection.retinanet_resnet50_fpn_v2(weights=weights)


### --- Modification for Custom Dataset ---
#### Get the number of input features for the classifier

In [3]:
num_anchors = model.head.classification_head.num_anchors
in_channels = model.backbone.out_channels

#### Define your number of classes (e.g., number of classes in your custom dataset + 1 for background)
 IMPORTANT: Torchvision RetinaNet usually expects num_classes = actual_classes + 1 (for background)
 However, documentation suggests the head should be built with num_classes = actual_classes * num_anchors.
 Double-check the specific documentation for the version you use, but often it's handled like this:

In [4]:
num_classes_custom = 2 # Replace with the actual number of classes in your dataset

In [5]:
new_cls_head = RetinaNetClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=num_classes_custom
)

In [6]:
# Replace the pre-trained head with the new one
model.head.classification_head = new_cls_head

# --- End Modification ---

In [8]:
# --- 1. Define Configuration ---
BATCH_SIZE = 8
ANNOTATIONS_DIR = "/work/mathiamt/SnowPoleDetection/torchVision/pytorch-retinanet/annotations"
CLASSES_FILE = "/work/mathiamt/SnowPoleDetection/torchVision/pytorch-retinanet/classes.csv"
# Define image directory - IMPORTANT: adjust this based on paths in your CSV
# If paths in CSV are like 'retinanet_data/lidar/images/train/image_1.png' and you run from project root:
IMG_DIR = '.'
# If paths in CSV are just 'image_1.png', use the full path to the specific train/valid directories
# IMG_DIR_TRAIN = '/work/mathiamt/SnowPoleDetection/torchVision/pytorch-retinanet/retinanet_data/lidar/images/train'
# IMG_DIR_VALID = '/work/mathiamt/SnowPoleDetection/torchVision/pytorch-retinanet/retinanet_data/lidar/images/valid'

# Get the transform functions
train_transforms = get_transforms(is_train=True)
val_transforms = get_transforms(is_train=False)

# --- 2. Create Dataset Instances ---
train_annotations_file = os.path.join(ANNOTATIONS_DIR, "lidar_train_annotations.csv")
val_annotations_file = os.path.join(ANNOTATIONS_DIR, "lidar_valid_annotations.csv")

# Make sure to provide all required arguments to LidarDataset:
# annotations_file, classes_file, img_dir, transforms
train_dataset = LidarDataset(
    annotations_file=train_annotations_file,
    classes_file=CLASSES_FILE,
    img_dir=None, # Or IMG_DIR_TRAIN if paths in CSV are just filenames
    transforms=train_transforms
)

val_dataset = LidarDataset(
    annotations_file=val_annotations_file,
    classes_file=CLASSES_FILE,
    img_dir=None, # Or IMG_DIR_VALID if paths in CSV are just filenames
    transforms=val_transforms
)

# --- 3. Create DataLoader Instances ---
# Now pass the dataset instances to the DataLoader
train_loader = DataLoader(
    dataset=train_dataset, # Pass the instantiated dataset
    batch_size=BATCH_SIZE,
    shuffle=True,          # Shuffle for training
    num_workers=2,         # Adjust based on your system
    collate_fn=collate_fn
)

val_loader = DataLoader(
    dataset=val_dataset,   # Pass the instantiated dataset
    batch_size=BATCH_SIZE,
    shuffle=False,         # No shuffle for validation
    num_workers=2,
    collate_fn=collate_fn
)

# --- Ready for Training ---
print(f"Created train_loader with {len(train_loader)} batches.")
print(f"Created val_loader with {len(val_loader)} batches.")

# You can now use train_loader and val_loader in your training loop

Created train_loader with 171 batches.
Created val_loader with 49 batches.


##### --- 1. Define the Model ---

In [9]:
# --- 2. Set the Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device) # Move model to the chosen device




Using device: cuda


RetinaNet(
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      

In [10]:
# --- 3. Define the Optimizer ---
# Get parameters that require gradients
params = [p for p in model.parameters() if p.requires_grad]

In [ ]:
# Choose optimizer and learning rate
learning_rate = 0.001
# optimizer = optim.Adam(params, lr=learning_rate, weight_decay=0.0001)
optimizer = optim.SGD(params, lr=learning_rate, momentum=0.9, weight_decay=0.0005) # SGD often used in papers

print("Model, Device, and Optimizer are set up.")

Model, Device, and Optimizer are set up.


In [20]:

# --- Training Configuration ---
num_epochs = 10 # Or choose desired number of epochs
output_dir = "checkpoints"
os.makedirs(output_dir, exist_ok=True) # Create directory if it doesn't exist
best_model_path = os.path.join(output_dir, "retinanet_best_map50.pth")
last_model_path = os.path.join(output_dir, "retinanet_last_epoch.pth")
# Optional: Learning Rate Scheduler
# Example: Reduce LR by a factor of 0.1 every 3 epochs
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# --- Instantiate the Metric ---
# We will calculate mAP using the COCO standard evaluation procedure
# It provides map, map_50, map_75, map_small, map_medium, map_large
metric = MeanAveragePrecision(iou_type="bbox")
# Move metric state to the correct device (important if using GPU)
metric.to(device)

best_map_50 = 0.0 # Initialize best mAP@0.95 score

# Lists to store history (optional)
train_loss_history = []
map_history = [] # Store mAP (0.50:0.95)
map_50_history = [] # Store mAP (0.50)

print("Starting Training...")
start_time = time.time()

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train() # Set model to training mode
    running_loss = 0.0
    epoch_train_loss = 0.0
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 10)

    # Iterate over data. Add tqdm here for a progress bar if desired:
    # from tqdm.notebook import tqdm
    # for batch_idx, (images, targets) in enumerate(tqdm(train_loader, desc=f"Training Epoch {epoch+1}")):
    for batch_idx, (images, targets) in enumerate(train_loader):
        # Move data to the correct device
        images = list(image.to(device) for image in images)
        # Targets is a list of dicts. Move tensors inside each dict to device.
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        # Torchvision detection models return a dict of losses during training
        loss_dict = model(images, targets)

        # Sum up the losses
        losses = sum(loss for loss in loss_dict.values())

        # Check for invalid loss values (NaN or Inf)
        if not torch.isfinite(losses):
            print(f"WARNING: Non-finite loss detected: {losses.item()}. Skipping batch {batch_idx}.")
            # Optionally: print loss_dict to investigate individual losses
            # print(loss_dict)
            continue # Skip optimizer step if loss is invalid

        # Backward pass
        losses.backward()

        # Optimizer step (update weights)
        optimizer.step()

        # --- Statistics ---
        current_loss = losses.item()
        running_loss += current_loss
        epoch_train_loss += current_loss

        # Print loss statistics every N batches (e.g., every 10 batches)
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(train_loader)} - Loss: {current_loss:.4f} (Avg: {running_loss/10:.4f})")
            # You can also print individual losses from loss_dict if needed:
            # loss_str = " ".join([f"{k}: {v.item():.4f}" for k, v in loss_dict.items()])
            # print(f"     Loss details: {loss_str}")
            running_loss = 0.0 # Reset running loss for the next N batches

    # Calculate average training loss for the epoch
    avg_epoch_train_loss = epoch_train_loss / len(train_loader)
    train_loss_history.append(avg_epoch_train_loss)
    print(f"Epoch {epoch+1} Training Loss: {avg_epoch_train_loss:.4f}")

    # Optional: Update the learning rate
    # if scheduler:
    #     scheduler.step()

    # --- Validation Phase ---
    print("\nRunning Validation and Calculating mAP...")
    model.eval() 
    with torch.no_grad(): 
        for images, targets in val_loader: # We still load targets for potential future evaluation
            images = list(image.to(device) for image in images)
           
            targets_cpu = [{k: v.cpu() for k, v in t.items()} for t in targets]
            
            # Get model predictions
            outputs = model(images)
            # Move predictions to CPU
            outputs_cpu = [{k: v.cpu() for k, v in t.items()} for t in outputs]

            # Update the metric state
            # Expected format: List[Dict[str, Tensor]] for both preds and targets
            # preds dict keys: 'boxes', 'scores', 'labels'
            # target dict keys: 'boxes', 'labels'
            metric.update(outputs_cpu, targets_cpu)

    # Compute the results over all batches
    map_50 = 0.0 # Default value in case of error
    map_val = 0.0
    try:
        results = metric.compute()
        map_50 = results['map_50'].item()
       
        map_val = results['map'].item() 
        map_history.append(map_val)
        map_50_history.append(map_50)

        print(f"Epoch {epoch+1} Validation Results:")
        print(f"  mAP@.50-.95: {map_val:.4f}")
        print(f"  mAP@.50:     {map_50:.4f}")


    except Exception as e:
        print(f"Could not compute mAP for epoch {epoch+1}: {e}")
        # Append placeholder values if computation failed
        map_history.append(0.0)
        map_50_history.append(0.0)


    # Reset the metric for the next epoch
    metric.reset()

    # --- Save Checkpoint Logic ---
    print(f"*** {map_50}: {best_map_50} ***")
    if map_50 > best_map_50:
        best_map_50 = map_50
        torch.save(model.state_dict(), best_model_path)
        print(f"*** New best model saved based on mAP@0.50: {best_map_50:.4f} at epoch {epoch+1} ***")
        print(f"*** Saved to: {best_model_path} ***")

    # Optionally, save the model from the very last epoch regardless of performance
    torch.save(model.state_dict(), last_model_path)
    print(f"Saved last epoch model state to: {last_model_path}") # Uncomment if needed


    print("Validation Phase Complete for Epoch", epoch+1)



# --- Training Complete ---
end_time = time.time()
total_time = end_time - start_time
print(f"\nTraining finished in {total_time // 60:.0f}m {total_time % 60:.0f}s")


#Optional: Plot training loss and mAP

fig, ax1 = plt.subplots()
color = 'tab:red'
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss', color=color)
ax1.plot(range(1, num_epochs + 1), train_loss_history, color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax2 = ax1.twinx() # instantiate a second axes that shares the same x-axis
color = 'tab:blue'
ax2.set_ylabel('mAP@.50', color=color) # we already handled the x-label with ax1
ax2.plot(range(1, num_epochs + 1), map_50_history, color=color, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)
fig.tight_layout() # otherwise the right y-label is slightly clipped
plt.title('Training Loss and mAP@.50 Over Epochs')
plt.show()

Starting Training...

Epoch 1/10
----------
  Batch 10/171 - Loss: 0.5882 (Avg: 0.6240)
  Batch 20/171 - Loss: 0.7097 (Avg: 0.6448)
  Batch 30/171 - Loss: 0.6666 (Avg: 0.6081)
  Batch 40/171 - Loss: 0.5543 (Avg: 0.5892)
  Batch 50/171 - Loss: 0.5447 (Avg: 0.5708)
  Batch 60/171 - Loss: 0.5103 (Avg: 0.6166)
  Batch 70/171 - Loss: 0.5935 (Avg: 0.6171)
  Batch 80/171 - Loss: 0.5889 (Avg: 0.6146)
  Batch 90/171 - Loss: 0.5321 (Avg: 0.6037)
  Batch 100/171 - Loss: 0.6935 (Avg: 0.5979)
  Batch 110/171 - Loss: 0.6470 (Avg: 0.6258)
  Batch 120/171 - Loss: 0.6683 (Avg: 0.6363)
  Batch 130/171 - Loss: 0.6013 (Avg: 0.5837)
  Batch 140/171 - Loss: 0.5496 (Avg: 0.5941)
  Batch 150/171 - Loss: 0.5948 (Avg: 0.6044)
  Batch 160/171 - Loss: 0.5850 (Avg: 0.5986)
  Batch 170/171 - Loss: 0.5816 (Avg: 0.6468)
Epoch 1 Training Loss: 0.6102

Running Validation and Calculating mAP...
Epoch 1 Validation Results:
  mAP@.50-.95: 0.1479
  mAP@.50:     0.5288
*** 0.5288031697273254: 0.0 ***
*** New best model save

KeyboardInterrupt: 